# Digital Twin 

In [22]:
pip install simpy

Note: you may need to restart the kernel to use updated packages.


# Data preprocessing and feature engineering

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from tensorflow.keras.preprocessing.sequence import pad_sequences
import random

# Load data
data = pd.read_csv('grocery_data_mostrecent.csv')
data['Date'] = pd.to_datetime(data['Date'], format='%d-%m-%Y')

# Generate synthetic Customer IDs
num_customers = 10  # Number of synthetic customers
data['CustomerID'] = np.random.randint(1, num_customers + 1, size=len(data))

# Extracting day of the week, and week of the year
data['DayOfWeek'] = data['Date'].dt.dayofweek
data['WeekOfYear'] = data['Date'].dt.isocalendar().week

# Initialize OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)  # Note the corrected parameter here

# Encode 'Item'
item_encoded = encoder.fit_transform(data[['Item']])
item_columns = [f'item_{category}' for category in encoder.categories_[0]]
data = pd.concat([data, pd.DataFrame(item_encoded, columns=item_columns, index=data.index)], axis=1)

# Encode 'Mode of Payment'
encoder = OneHotEncoder(sparse_output=False)
payment_encoded = encoder.fit_transform(data[['Mode of Payment']])
payment_columns = [f'payment_{method}' for method in encoder.categories_[0]]
data = pd.concat([data, pd.DataFrame(payment_encoded, columns=payment_columns, index=data.index)], axis=1)

# Normalize continuous data
data['Normalized_Price'] = (data['Price'] - data['Price'].mean()) / data['Price'].std()
data['Normalized_Quantity'] = (data['Quantity'] - data['Quantity'].mean()) / data['Quantity'].std()

# Display some of the data to verify
print(data.head())


        Date    Item  Price  Quantity Mode of Payment  CustomerID  DayOfWeek  \
0 2022-01-04    eggs   2.19         1      Debit Card           5          1   
1 2022-01-07    eggs   2.19         2            Cash           6          4   
2 2022-01-07   bread   2.99         3            Cash           2          4   
3 2022-01-07    rice   5.99         1            Cash           3          4   
4 2022-01-08  yogurt   1.99         1            Cash           6          5   

   WeekOfYear  item_almond extract  item_apples  ...  item_vinegar  \
0           1                  0.0          0.0  ...           0.0   
1           1                  0.0          0.0  ...           0.0   
2           1                  0.0          0.0  ...           0.0   
3           1                  0.0          0.0  ...           0.0   
4           1                  0.0          0.0  ...           0.0   

   item_waffle mix  item_water  item_yogurt  item_zucchini  payment_Cash  \
0              0.0    

# GAN Architecture Setup

In [39]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# Load data and preprocess
data = pd.read_csv('grocery_data_mostrecent.csv')
data['Date'] = pd.to_datetime(data['Date'], format='%d-%m-%Y')

# Generate synthetic customer IDs
data['CustomerID'] = np.random.randint(1000, size=len(data))

# One-hot encode the 'Item' column using OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)  # Use sparse_output=False instead of sparse=False
item_encoded = encoder.fit_transform(data[['Item']])

# Check if the newer method is available, otherwise use the older method
if hasattr(encoder, 'get_feature_names_out'):
    item_columns = encoder.get_feature_names_out(input_features=['Item'])
else:
    # Older versions of sklearn: Create feature names manually
    item_columns = ['item_' + feature for feature in encoder.get_feature_names()]
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Function to build the discriminator with adjustments
def build_discriminator(input_shape):
    model = Sequential([
        Flatten(input_shape=input_shape),
        Dense(512, activation=LeakyReLU(alpha=0.2), kernel_regularizer=l2(0.01)),
        Dropout(0.3),
        Dense(256, activation=LeakyReLU(alpha=0.2), kernel_regularizer=l2(0.01)),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    return model

# Compile the discriminator with a slightly modified optimizer
discriminator = build_discriminator(input_shape=(20, 124))
discriminator.compile(optimizer=Adam(0.0001, 0.9), loss='binary_crossentropy', metrics=['accuracy'])

# Add encoded features to DataFrame
data = pd.concat([data, pd.DataFrame(item_encoded, columns=item_columns, index=data.index)], axis=1)

# Group by CustomerID and create a list of purchases
grouped = data.groupby('CustomerID')[item_columns].apply(lambda x: x.values.tolist())

# Convert grouped data to a list of lists suitable for padding
sequences = list(grouped)

# Pad sequences to ensure consistent length
padded_sequences = pad_sequences(sequences, maxlen=20, padding='post', dtype='float32')


In [40]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Function to build the discriminator with adjustments
def build_discriminator(input_shape):
    model = Sequential([
        Flatten(input_shape=input_shape),
        Dense(512, activation=LeakyReLU(alpha=0.2), kernel_regularizer=l2(0.01)),
        Dropout(0.3),
        Dense(256, activation=LeakyReLU(alpha=0.2), kernel_regularizer=l2(0.01)),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    return model

# Compile the discriminator with a slightly modified optimizer
discriminator = build_discriminator(input_shape=(20, 124))
discriminator.compile(optimizer=Adam(0.0001, 0.9), loss='binary_crossentropy', metrics=['accuracy'])


# Function to build the generator
def build_generator(latent_dim, output_dim):
    model = Sequential([
        Dense(256, input_dim=latent_dim, activation='relu'),
        LeakyReLU(alpha=0.2),
        Dense(512, activation='relu'),
        LeakyReLU(alpha=0.2),
        Dense(1024, activation='relu'),
        LeakyReLU(alpha=0.2),
        Dense(output_dim, activation='tanh'),
        Reshape((20, 124))  # Ensure this reshape matches the new discriminator input shape
    ])
    return model

# Define the latent dimension and output dimension
latent_dim = 100
output_dim = 20 * 124  # This should be product of the dimensions expected by discriminator

# Instantiate the generator and discriminator
generator = build_generator(latent_dim, output_dim)
discriminator = build_discriminator(input_shape=(20, 124))

# Compile the discriminator
discriminator.compile(optimizer=Adam(0.0002, 0.5), loss='binary_crossentropy', metrics=['accuracy'])



In [41]:
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_gan(generator, discriminator):
    # Ensure the discriminator's trainable status is correctly set before compiling the GAN
    discriminator.trainable = False  # Freeze the discriminator when training the GAN
    gan_input = Input(shape=(latent_dim,))
    gan_output = discriminator(generator(gan_input))
    gan = Model(inputs=gan_input, outputs=gan_output)
    gan.compile(optimizer=Adam(0.0002, 0.5), loss='binary_crossentropy')
    return gan

# Ensure you define generator and discriminator before this
gan = build_gan(generator, discriminator)


In [42]:
print("Generator output shape:", generator.output_shape)
print("Discriminator input shape:", discriminator.input_shape)


Generator output shape: (None, 20, 124)
Discriminator input shape: (None, 20, 124)


In [43]:
def train_gan(generator, discriminator, gan, sequences, batch_size=32, epochs=100):
    latent_dim = 100  # This should be set as per your GAN architecture

    for epoch in range(epochs):
        noise = np.random.normal(0, 1, (batch_size, latent_dim))

        for batch_start in range(0, sequences.shape[0], batch_size):
            real_seqs = sequences[batch_start:batch_start + batch_size]

            # Check the actual batch size
            actual_batch_size = real_seqs.shape[0]
            if actual_batch_size == 0:
                continue

            # Regenerate noise and adjust label sizes for actual batch size
            noise = np.random.normal(0, 1, (actual_batch_size, latent_dim))
            fake_seqs = generator.predict(noise)

            real_y = np.ones((actual_batch_size, 1))
            fake_y = np.zeros((actual_batch_size, 1))

            discriminator.trainable = True
            d_loss_real = discriminator.train_on_batch(real_seqs, real_y)
            d_loss_fake = discriminator.train_on_batch(fake_seqs, fake_y)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # Train generator
            discriminator.trainable = False
            g_loss = gan.train_on_batch(noise, np.ones((actual_batch_size, 1)))

            print(f"Epoch {epoch + 1}/{epochs}, Discriminator Loss: {d_loss}, Generator Loss: {g_loss}")


In [44]:
# Make sure these are set correctly once before you start training
discriminator.trainable = True  # Trainable for standalone training
discriminator.compile(optimizer='adam', loss='binary_crossentropy')

discriminator.trainable = False  # Not trainable when part of GAN
gan.compile(optimizer='adam', loss='binary_crossentropy')

# Now you can train your GAN
train_gan(generator, discriminator, gan, corrected_sequences, 32, 50)


1/1 [==============================] - 0s 73ms/step
Epoch 1/50, Discriminator Loss: 12.264902591705322, Generator Loss: 12.747379302978516
1/1 [==============================] - 0s 58ms/step
Epoch 1/50, Discriminator Loss: 11.453659057617188, Generator Loss: 10.709990501403809
1/1 [==============================] - 0s 59ms/step
Epoch 1/50, Discriminator Loss: 10.678382396697998, Generator Loss: 10.091180801391602
1/1 [==============================] - 0s 44ms/step
Epoch 1/50, Discriminator Loss: 9.72295331954956, Generator Loss: 10.143375396728516
1/1 [==============================] - 0s 35ms/step
Epoch 1/50, Discriminator Loss: 8.867437362670898, Generator Loss: 10.707633972167969
1/1 [==============================] - 0s 28ms/step
Epoch 1/50, Discriminator Loss: 8.213206052780151, Generator Loss: 10.421895980834961
1/1 [==============================] - 0s 31ms/step
Epoch 1/50, Discriminator Loss: 7.638379812240601, Generator Loss: 9.992982864379883
1/1 [============================

In [45]:
import numpy as np

def adjust_sequence_length(sequences, target_length, feature_dim):
    adjusted_sequences = []
    for seq in sequences:
        if len(seq) > target_length:
            adjusted_seq = seq[:target_length]
        else:
            padding = np.zeros((target_length - len(seq), feature_dim))
            adjusted_seq = np.vstack((seq, padding))
        adjusted_sequences.append(adjusted_seq)
    return np.array(adjusted_sequences)

def train_gan(generator, discriminator, gan, sequences, batch_size=32, epochs=100):
    latent_dim = 100  # Adjust as per your generator's input requirement
    
    for epoch in range(epochs):
        for batch_start in range(0, sequences.shape[0], batch_size):
            real_seqs = sequences[batch_start:batch_start + batch_size]
            actual_batch_size = real_seqs.shape[0]  # Handle variable batch sizes
            noise = np.random.normal(0, 1, (actual_batch_size, latent_dim))

            fake_seqs = generator.predict(noise)

            real_y = np.ones((actual_batch_size, 1))
            fake_y = np.zeros((actual_batch_size, 1))

            discriminator.trainable = True
            d_loss_real = discriminator.train_on_batch(real_seqs, real_y)
            d_loss_fake = discriminator.train_on_batch(fake_seqs, fake_y)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            discriminator.trainable = False
            g_loss = gan.train_on_batch(noise, np.ones((actual_batch_size, 1)))

            print(f"Epoch {epoch + 1}/{epochs}, Discriminator Loss: {d_loss}, Generator Loss: {g_loss}")

# Correct sequence lengths as needed
corrected_sequences = adjust_sequence_length(padded_sequences, 20, 124)

# Start training the GAN
train_gan(generator, discriminator, gan, corrected_sequences, 32, 50)


1/1 [==============================] - 0s 48ms/step
Epoch 1/50, Discriminator Loss: 0.11467799544334412, Generator Loss: 5.517945766448975
1/1 [==============================] - 0s 36ms/step
Epoch 1/50, Discriminator Loss: 0.11171317100524902, Generator Loss: 5.656230926513672
1/1 [==============================] - 0s 35ms/step
Epoch 1/50, Discriminator Loss: 0.10892355442047119, Generator Loss: 5.811424255371094
1/1 [==============================] - 0s 26ms/step
Epoch 1/50, Discriminator Loss: 0.10373799875378609, Generator Loss: 5.6785359382629395
1/1 [==============================] - 0s 24ms/step
Epoch 1/50, Discriminator Loss: 0.10031575337052345, Generator Loss: 5.276027202606201
1/1 [==============================] - 0s 27ms/step
Epoch 1/50, Discriminator Loss: 0.09752319008111954, Generator Loss: 5.675417423248291
1/1 [==============================] - 0s 52ms/step
Epoch 1/50, Discriminator Loss: 0.09964229539036751, Generator Loss: 5.836763381958008
1/1 [=====================

In [46]:
from tensorflow.keras.optimizers import Adam

# Function to update the learning rate directly
def update_learning_rate(optimizer, epoch):
    initial_lr = 0.0001
    if epoch < 10:
        lr = initial_lr
    else:
        lr = initial_lr * np.exp(-0.1 * (epoch - 10))
    optimizer.lr.assign(lr)

# Training function with manual learning rate updates
def train_gan(generator, discriminator, gan, sequences, batch_size=32, epochs=100):
    latent_dim = 100  # Ensure this matches your GAN architecture
    optimizer_discriminator = Adam(0.0001)
    optimizer_gan = Adam(0.0001)

    discriminator.compile(optimizer=optimizer_discriminator, loss='binary_crossentropy', metrics=['accuracy'])
    gan.compile(optimizer=optimizer_gan, loss='binary_crossentropy')

    for epoch in range(epochs):
        # Update learning rates
        update_learning_rate(optimizer_discriminator, epoch)
        update_learning_rate(optimizer_gan, epoch)

        for batch_start in range(0, len(sequences), batch_size):
            real_seqs = sequences[batch_start:batch_start + batch_size]
            if len(real_seqs) == 0:
                continue

            noise = np.random.normal(0, 1, (len(real_seqs), latent_dim))
            fake_seqs = generator.predict(noise)

            real_y = np.ones((len(real_seqs), 1))
            fake_y = np.zeros((len(real_seqs), 1))

            # Train discriminator
            discriminator.trainable = True
            d_loss_real = discriminator.train_on_batch(real_seqs, real_y)
            d_loss_fake = discriminator.train_on_batch(fake_seqs, fake_y)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # Train generator
            discriminator.trainable = False
            g_loss = gan.train_on_batch(noise, np.ones((len(real_seqs), 1)))

        print(f"Epoch {epoch + 1}/{epochs}, Discriminator Loss: {d_loss}, Generator Loss: {g_loss}")

# Ensure sequences are correctly shaped and call the training function
train_gan(generator, discriminator, gan, np.array(padded_sequences), 32, 50)


1/1 [==============================] - 0s 33ms/step
Epoch 1/50, Discriminator Loss: [0.64586103 0.64285715], Generator Loss: 0.7823058366775513
1/1 [==============================] - 0s 41ms/step
Epoch 2/50, Discriminator Loss: [1.47538377 0.5       ], Generator Loss: 0.24015219509601593
1/1 [==============================] - 0s 46ms/step
Epoch 3/50, Discriminator Loss: [1.75981794 0.5       ], Generator Loss: 0.18676576018333435
1/1 [==============================] - 0s 36ms/step
Epoch 4/50, Discriminator Loss: [2.12394112 0.5       ], Generator Loss: 0.17463719844818115
1/1 [==============================] - 0s 34ms/step
Epoch 5/50, Discriminator Loss: [2.21489354 0.5       ], Generator Loss: 0.17845913767814636
1/1 [==============================] - 0s 35ms/step
Epoch 6/50, Discriminator Loss: [2.13832124 0.5       ], Generator Loss: 0.16271793842315674
1/1 [==============================] - 0s 42ms/step
Epoch 7/50, Discriminator Loss: [2.37134308 0.5       ], Generator Loss: 0.1698